In [ ]:
# ============================================================
# 08 / NEW CELL
# Advanced basemap figure:
# mismatch direction with real basemap + focus inset panels
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import contextily as ctx

# ------------------------------------------------------------
# project root
# ------------------------------------------------------------
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "requirements.txt").exists() and (PROJECT_ROOT.parent / "requirements.txt").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

FIG_DIR = PROJECT_ROOT / "figures" / "pilot"
OUT_DIR = PROJECT_ROOT / "outputs" / "pilot"
FIG_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)

# ------------------------------------------------------------
# config
# ------------------------------------------------------------
CITY_CFG = {
    "Houston": {
        "path": PROJECT_ROOT / "data_processed/houston/houston_master_with_lst_hi_fixed.gpkg",
        "lst_col": "lst_c",
        "hi_col": "hi_c",
        "geoid_col": "GEOID",
        "focus_buffer_m": 18000,
    },
    "Phoenix": {
        "path": PROJECT_ROOT / "data_processed/phoenix/phoenix_master_with_lst_hi_fixed.gpkg",
        "lst_col": "lst_c",
        "hi_col": "hi_c",
        "geoid_col": "GEOID",
        "focus_buffer_m": 14000,
    },
}

# ------------------------------------------------------------
# helpers
# ------------------------------------------------------------
def safe_zscore(series):
    s = pd.to_numeric(series, errors="coerce")
    std = s.std(ddof=0)
    if pd.isna(std) or std == 0:
        return pd.Series(np.zeros(len(s)), index=s.index)
    return (s - s.mean()) / std

def classify_gap(g):
    if pd.isna(g):
        return "Missing"
    if g >= 1.0:
        return "Strong HI > LST"
    elif g >= 0.3:
        return "Mild HI > LST"
    elif g > -0.3:
        return "Equal"
    elif g > -1.0:
        return "Mild LST > HI"
    else:
        return "Strong LST > HI"

GAP_COLORS = {
    "Strong HI > LST": "#b2182b",
    "Mild HI > LST": "#ef8a62",
    "Equal": "#f7f7f7",
    "Mild LST > HI": "#67a9cf",
    "Strong LST > HI": "#2166ac",
    "Missing": "#d9d9d9",
}

PLOT_ORDER = [
    "Missing",
    "Equal",
    "Mild LST > HI",
    "Strong LST > HI",
    "Mild HI > LST",
    "Strong HI > LST",
]

def add_north_arrow(ax, x=0.92, y=0.82, dy=0.12):
    ax.annotate(
        "",
        xy=(x, y),
        xytext=(x, y - dy),
        xycoords="axes fraction",
        arrowprops=dict(facecolor="black", edgecolor="black", width=2, headwidth=10, headlength=10)
    )
    ax.text(
        x, y + 0.02, "N",
        transform=ax.transAxes,
        ha="center", va="bottom",
        fontsize=12, fontweight="bold"
    )

def add_scalebar(ax, length_m=10000, label="10 km", lw=2):
    x0, x1 = ax.get_xlim()
    y0, y1 = ax.get_ylim()
    width = x1 - x0
    height = y1 - y0

    bar_x = x0 + 0.06 * width
    bar_y = y0 + 0.07 * height

    ax.plot([bar_x, bar_x + length_m], [bar_y, bar_y], color="black", lw=lw)
    ax.plot([bar_x, bar_x], [bar_y - 0.015 * height, bar_y + 0.015 * height], color="black", lw=lw)
    ax.plot([bar_x + length_m, bar_x + length_m], [bar_y - 0.015 * height, bar_y + 0.015 * height], color="black", lw=lw)
    ax.text(bar_x + length_m / 2, bar_y + 0.025 * height, label, ha="center", va="bottom", fontsize=10)

def plot_gap_map(ax, gdf, title, zoom_bounds=None):
    # plot basemap first via extent
    if zoom_bounds is None:
        xmin, ymin, xmax, ymax = gdf.total_bounds
    else:
        xmin, ymin, xmax, ymax = zoom_bounds

    xpad = (xmax - xmin) * 0.05
    ypad = (ymax - ymin) * 0.05
    ax.set_xlim(xmin - xpad, xmax + xpad)
    ax.set_ylim(ymin - ypad, ymax + ypad)

    # basemap
    ctx.add_basemap(
        ax,
        source=ctx.providers.CartoDB.Positron,
        crs=gdf.crs,
        attribution_size=6
    )

    # polygons
    for cls in PLOT_ORDER:
        sub = gdf[gdf["gap_class"] == cls]
        if len(sub) == 0:
            continue
        sub.plot(
            ax=ax,
            color=GAP_COLORS[cls],
            edgecolor="white",
            linewidth=0.20,
            alpha=0.72
        )

    # outline
    outline = gpd.GeoDataFrame(geometry=[gdf.unary_union], crs=gdf.crs)
    outline.boundary.plot(ax=ax, color="black", linewidth=0.9)

    ax.set_title(title, fontsize=15, pad=8)
    ax.set_axis_off()
    add_north_arrow(ax)
    add_scalebar(ax)

# ------------------------------------------------------------
# load and prepare city data
# ------------------------------------------------------------
city_data2 = {}
summary_rows = []

for city, cfg in CITY_CFG.items():
    gdf = gpd.read_file(cfg["path"]).copy()
    gdf = gdf.loc[gdf.geometry.notna()].copy()

    lst_col = cfg["lst_col"]
    hi_col = cfg["hi_col"]

    gdf[lst_col] = pd.to_numeric(gdf[lst_col], errors="coerce")
    gdf[hi_col] = pd.to_numeric(gdf[hi_col], errors="coerce")

    valid = gdf.loc[gdf[lst_col].notna() & gdf[hi_col].notna()].copy()

    valid["lst_z"] = safe_zscore(valid[lst_col])
    valid["hi_z"] = safe_zscore(valid[hi_col])
    valid["gap_z"] = valid["hi_z"] - valid["lst_z"]

    gdf["gap_z"] = np.nan
    gdf.loc[valid.index, "gap_z"] = valid["gap_z"]
    gdf["gap_class"] = gdf["gap_z"].apply(classify_gap)

    # project for basemap
    if gdf.crs is None:
        raise ValueError(f"{city} CRS is missing")
    if gdf.crs.to_string() != "EPSG:3857":
        gdf = gdf.to_crs(epsg=3857)

    # choose focus area automatically:
    # take top 12% tracts by abs(gap_z), then get centroid of their union
    valid_3857 = gdf.loc[gdf["gap_z"].notna()].copy()
    cutoff = valid_3857["gap_z"].abs().quantile(0.88)
    extreme = valid_3857.loc[valid_3857["gap_z"].abs() >= cutoff].copy()

    if len(extreme) == 0:
        focus_geom = valid_3857.unary_union.centroid
    else:
        focus_geom = extreme.unary_union.centroid

    buf = cfg["focus_buffer_m"]
    focus_bounds = focus_geom.buffer(buf).bounds  # xmin, ymin, xmax, ymax

    city_data2[city] = {
        "gdf": gdf,
        "focus_bounds": focus_bounds,
        "n_total": len(gdf),
        "n_valid": int(gdf["gap_z"].notna().sum()),
        "n_extreme": int(len(extreme)),
        "focus_x": focus_geom.x,
        "focus_y": focus_geom.y,
    }

    vc = gdf["gap_class"].value_counts()
    summary_rows.append({
        "city": city,
        "n_total": len(gdf),
        "n_valid": int(gdf["gap_z"].notna().sum()),
        "strong_hi_gt_lst_n": int(vc.get("Strong HI > LST", 0)),
        "mild_hi_gt_lst_n": int(vc.get("Mild HI > LST", 0)),
        "equal_n": int(vc.get("Equal", 0)),
        "mild_lst_gt_hi_n": int(vc.get("Mild LST > HI", 0)),
        "strong_lst_gt_hi_n": int(vc.get("Strong LST > HI", 0)),
        "missing_n": int(vc.get("Missing", 0)),
        "focus_buffer_m": buf,
    })

gap_focus_summary = pd.DataFrame(summary_rows)
display(gap_focus_summary)

gap_focus_summary_out = OUT_DIR / "rq1_gap_focus_basemap_summary.csv"
gap_focus_summary.to_csv(gap_focus_summary_out, index=False)
print(gap_focus_summary_out, "->", gap_focus_summary_out.exists())

# ------------------------------------------------------------
# plot figure: main panels + focus inset panels
# ------------------------------------------------------------
fig = plt.figure(figsize=(16, 12))
gs = fig.add_gridspec(2, 2, hspace=0.12, wspace=0.06)

ax1 = fig.add_subplot(gs[0, 0])
ax2 = fig.add_subplot(gs[0, 1])
ax3 = fig.add_subplot(gs[1, 0])
ax4 = fig.add_subplot(gs[1, 1])

plot_gap_map(ax1, city_data2["Houston"]["gdf"], "A. Houston: citywide mismatch direction")
plot_gap_map(ax2, city_data2["Phoenix"]["gdf"], "B. Phoenix: citywide mismatch direction")
plot_gap_map(ax3, city_data2["Houston"]["gdf"], "C. Houston: focus area", zoom_bounds=city_data2["Houston"]["focus_bounds"])
plot_gap_map(ax4, city_data2["Phoenix"]["gdf"], "D. Phoenix: focus area", zoom_bounds=city_data2["Phoenix"]["focus_bounds"])

legend_handles = [
    Patch(facecolor=GAP_COLORS["Strong HI > LST"], edgecolor="black", linewidth=0.2, label="Strong HI > LST"),
    Patch(facecolor=GAP_COLORS["Mild HI > LST"], edgecolor="black", linewidth=0.2, label="Mild HI > LST"),
    Patch(facecolor=GAP_COLORS["Equal"], edgecolor="black", linewidth=0.2, label="Equal"),
    Patch(facecolor=GAP_COLORS["Mild LST > HI"], edgecolor="black", linewidth=0.2, label="Mild LST > HI"),
    Patch(facecolor=GAP_COLORS["Strong LST > HI"], edgecolor="black", linewidth=0.2, label="Strong LST > HI"),
    Patch(facecolor=GAP_COLORS["Missing"], edgecolor="black", linewidth=0.2, label="Missing / no data"),
]

fig.legend(
    handles=legend_handles,
    loc="lower center",
    ncol=6,
    frameon=False,
    fontsize=10,
    bbox_to_anchor=(0.5, 0.03)
)

fig.suptitle(
    "RQ1 deep-dive: LST–HI mismatch direction on real basemap",
    fontsize=19,
    y=0.98
)

fig.text(
    0.5, 0.008,
    "Red indicates tracts where Heat Index is relatively stronger than Land Surface Temperature. "
    "Blue indicates tracts where LST is relatively stronger. "
    "Bottom panels zoom into the most spatially concentrated mismatch areas.",
    ha="center",
    fontsize=10
)

basemap_fig_out = FIG_DIR / "rq1_gap_direction_basemap_focus.png"
fig.savefig(basemap_fig_out, dpi=300, bbox_inches="tight", facecolor="white")
plt.show()

print(basemap_fig_out, "->", basemap_fig_out.exists())